In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

path = "C:\\Users\\User0\\PycharmProjects\\zoomcamp-hw\\homeworks\\02-regression\\data\\car_fuel_efficiency_2026.csv"

df = pd.read_csv(path)

In [60]:
df.shape

(10000, 11)

In [61]:
df.columns.tolist()
features = ["engine_displacement","horsepower","vehicle_weight","model_year","fuel_efficiency_mpg"]
df_sub = df[features]
df_sub.shape

(10000, 5)

In [62]:
df_sub.fuel_efficiency_mpg.tail()

9995    27.4
9996    30.0
9997    28.1
9998    32.2
9999    30.2
Name: fuel_efficiency_mpg, dtype: float64

In [63]:
df_sub.isna().sum().sort_values(ascending=False)

horsepower             877
engine_displacement      0
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [64]:
df_sub.describe()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
count,10000.000000,9123.000000,10000.000000,10000.000000,10000.000000
mean,2268.807000,254.446016,4286.754000,1999.549500,29.997700
std,183.447355,22.844613,266.212491,14.207925,2.930245
min,1590.000000,171.000000,3320.000000,1975.000000,19.800000
25%,2150.000000,239.000000,4110.000000,1987.000000,28.000000
50%,2270.000000,254.000000,4280.000000,2000.000000,30.000000
75%,2390.000000,270.000000,4470.000000,2012.000000,31.900000
max,3110.000000,334.000000,5460.000000,2024.000000,41.200000


For question 3, I will fill the missing values in the horsepower column with the constant values(0).For this version I will train linear regression without regularization. After training I will compute the rmse, and then I will train linear regression also without regularization but with the missing values filled with the mean values of the column. The goal is to compate the rmse of the two models and see which one performs better.

In [65]:
cols = ["engine_displacement","horsepower","vehicle_weight","model_year","fuel_efficiency_mpg"]
df = df[cols]

In [66]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [67]:
features = ["engine_displacement","horsepower","vehicle_weight","model_year"]
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values


In [68]:
def linear_regression(X,y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones,X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    w0 = w_full[0]
    w = w_full[1:]
    return w0,w

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [69]:
def make_X(df,fill_value):
    df_copy = df.copy()
    df_copy["horsepower"] = df_copy["horsepower"].fillna(fill_value)
    return df_copy[features].values

def evaluate(fill_value):
    X_train = make_X(df_train,fill_value)
    X_val = make_X(df_val,fill_value)

    w0,w = linear_regression(X_train,y_train)
    y_pred = w0 + X_val.dot(w)
    return rmse(y_val,y_pred)

In [70]:
score_zero = evaluate(0)
mean_hp = df_train.horsepower.mean()
score_mean = evaluate(mean_hp)
print(round(score_zero,3))
print(round(score_mean,3))

2.205
2.202


In [71]:
def linear_regression_reg(X,y,r=0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones,X])

    XTX = X.T.dot(X)
    XTX = XTX +r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    w0 = w_full[0]
    w = w_full[1:]
    return w0,w

In [72]:
def evaluate_reg(fill_value,r):
    X_train = make_X(df_train,fill_value)
    X_val = make_X(df_val,fill_value)

    w0,w = linear_regression_reg(X_train,y_train,r)
    y_pred = w0 + X_val.dot(w)
    return rmse(y_val,y_pred)

In [73]:
for r in [0, 0.01, 0.1, 1, 5, 10,100]:
    score_zero = evaluate_reg(0,r)
    print(r,round(score_zero,4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


Question 5 solution

In [74]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

In [75]:
def split_data(df,seed):
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]
    return df_train,df_val,df_test

In [76]:
scores = []
for seed in [0,1,2,3,4,5,6,7,8,9]:
    df_train,df_val,df_test = split_data(df,seed)

    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values

    score_zero = evaluate(0)
    scores.append(score_zero)
std = np.std(scores)
print(round(std,3))

0.029


q6

In [77]:
df_train,df_val,df_test = split_data(df,9)

df_train_full = pd.concat([df_train,df_val])

X_train_full = make_X(df_train_full,0)
X_test = make_X(df_test,0)

y_train_full = df_train_full.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values


In [78]:
w0, w = linear_regression_reg(X_train_full, y_train_full, r=0.001)


y_pred = w0 + X_test.dot(w)

score = rmse(y_test,y_pred)
print(round(score,3))

2.236
